<a href="https://colab.research.google.com/github/sultanjacob/LondonBoroughsDumpWatch/blob/main/Boroughs_dump_watch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LondonBoroughsDumpWatch: Predictive Geospatial ML Framework for Urban Waste Enforcement Optimization
## Phase 1: Spatial Grid Indexing & Proximity Feature Engineering

### 1. Executive Summary
Illegal dumping (fly-tipping) is a significant environmental and economic drain on local government resources. London's unique layout—comprising 32 distinct boroughs and high-density urban transit nodes—creates intricate spatial variations in waste accumulation.

Traditional municipal responses are reactive, deploying cleanup crews *after* incidents are reported. **LondonBoroughsDumpWatch** transforms this workflow into a proactive strategy. By combining spatial machine learning with geographic indexing, this framework maps out localized risk surfaces to help councils allocate enforcement resources and optimize cleaning schedules *before* dumping occurs.

### 2. Analytical Objectives
* **Eliminate Spatial Boundary Bias:** Replace irregular borough administration lines with an optimized, uniform spatial hexagonal grid system (**Uber H3 Index**).
* **Quantify Spatial Proximity Dependencies:** Calculate high-performance geodesic distance maps from potential hotspots to critical urban infrastructure points (e.g., public transit junctions, recycling facilities).
* **Engineer the Modeling Matrix:** Aggregate localized incident frequencies into tabular feature structures ready for supervised spatial classification models.

### 3. Pipeline Architecture
1. **Environment Setup & Spatial Contextualisation:** Initialize robust geographic libraries (`geopandas`, `h3`, `shapely`).
2. **Spatial Layer Synthesis:** Emulate time-stamped incident coordinates mapped to realistic London spatial bounds, factoring in point-pattern clustering behaviors.
3. **Hexagonal Grid Mapping:** Tessellate coordinates into localized spatial bins at **H3 Resolution 8** (approx. 0.7 square km area per cell).
4. **Vectorised Proximity Computation:** Leverage a spatial `BallTree` index structure to calculate high-speed Haversine distances to critical infrastructure.


In [2]:
# Install specialized geospatial indexing libraries missing from the default Colab runtime
!pip install h3 geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.4 MB/s  0:00:00


In [3]:
# ==============================================================================
# 1. Environment Setup & Core Pipeline Implementation
# ==============================================================================
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import h3
from sklearn.neighbors import BallTree

def generate_synthetic_london_data(num_incidents=2500, num_hubs=15, num_recycling=6):
    """
    Simulates high-fidelity geospatial incident points and infrastructure layers
    anchored accurately inside realistic London lat/lon boundaries.
    """
    np.random.seed(42)

    # Approximate bounding coordinates for Greater London
    lat_min, lat_max = 51.35, 51.65
    lon_min, lon_max = -0.45, 0.25

    # Generate background uniform distribution of fly-tipping events
    lats = np.random.uniform(lat_min, lat_max, num_incidents)
    lons = np.random.uniform(lon_min, lon_max, num_incidents)

    # Superimpose high-density clusters to emulate repetitive historical dumping hotspots
    num_clusters = 6
    for _ in range(num_clusters):
        center_lat = np.random.uniform(lat_min, lat_max)
        center_lon = np.random.uniform(lon_min, lon_max)

        # Pull a random subset of indexes to force into a tight Gaussian cluster
        cluster_indices = np.random.choice(num_incidents, num_incidents // 8, replace=False)
        lats[cluster_indices] = np.random.normal(center_lat, 0.015, len(cluster_indices))
        lons[cluster_indices] = np.random.normal(center_lon, 0.015, len(cluster_indices))

    df_incidents = pd.DataFrame({'latitude': lats, 'longitude': lons})
    df_incidents['timestamp'] = pd.to_datetime(
        np.random.choice(pd.date_range('2026-01-01', '2026-10-31'), num_incidents)
    )

    # Simulate infrastructure locations (features that affect dumping probability)
    df_hubs = pd.DataFrame({
        'hub_id': [f"TRANSIT_HUB_{i}" for i in range(num_hubs)],
        'lat': np.random.uniform(lat_min, lat_max, num_hubs),
        'lon': np.random.uniform(lon_min, lon_max, num_hubs)
    })

    df_recycling = pd.DataFrame({
        'center_id': [f"RECYCLING_CTR_{i}" for i in range(num_recycling)],
        'lat': np.random.uniform(lat_min, lat_max, num_recycling),
        'lon': np.random.uniform(lon_min, lon_max, num_recycling)
    })

    return df_incidents, df_hubs, df_recycling

def build_h3_grid_features(df_incidents, df_hubs, df_recycling, h3_resolution=8):
    """
    Transforms point vector geometries into a discrete hexagonal feature matrix,
    calculating geographical proximity fields via high-efficiency lookup indexing.
    """
    # Step A: Compute specific H3 index mapping for every point coordinate pair
    df_incidents['h3_index'] = df_incidents.apply(
        lambda row: h3.geo_to_h3(row['latitude'], row['longitude'], h3_resolution), axis=1
    )

    # Step B: Aggregate total raw incident counts within each unique hexagon cell
    grid_matrix = df_incidents.groupby('h3_index').size().reset_index(name='incident_count')

    # Step C: Extract geographical centroids for each grid element
    centroids = grid_matrix['h3_index'].apply(lambda h: h3.h3_to_geo(h))
    grid_matrix['centroid_lat'] = [c[0] for c in centroids]
    grid_matrix['centroid_lon'] = [c[1] for c in centroids]

    # Step D: Construct a high-performance BallTree utilizing angular Radian units
    hub_coords_rad = np.radians(df_hubs[['lat', 'lon']].values)
    rec_coords_rad = np.radians(df_recycling[['lat', 'lon']].values)
    centroid_coords_rad = np.radians(grid_matrix[['centroid_lat', 'centroid_lon']].values)

    tree_hubs = BallTree(hub_coords_rad, metric='haversine')
    tree_rec = BallTree(rec_coords_rad, metric='haversine')

    # Step E: Query indices for closest elements and scale output by Earth's radius (6371km)
    dist_hubs, _ = tree_hubs.query(centroid_coords_rad, k=1)
    dist_rec, _ = tree_rec.query(centroid_coords_rad, k=1)

    grid_matrix['dist_to_transit_hub_km'] = dist_hubs * 6371
    grid_matrix['dist_to_recycling_km'] = dist_rec * 6371

    # Step F: Establish the supervised classification ground-truth vector
    # 1 indicates target dumping frequency sits over the regional median value (High Risk Zone)
    median_threshold = grid_matrix['incident_count'].median()
    grid_matrix['high_risk_target'] = (grid_matrix['incident_count'] > median_threshold).astype(int)

    return grid_matrix

# Execute simulation configuration pipeline
raw_incidents, transit_hubs, recycling_points = generate_synthetic_london_data()
processed_feature_matrix = build_h3_grid_features(raw_incidents, transit_hubs, recycling_points)

# Display the engineered dataset schema structure
print(f"Data Engineering Phase Complete! Unique Engineered Spatial Bins (Rows): {processed_feature_matrix.shape[0]}")
processed_feature_matrix.head()


AttributeError: module 'h3' has no attribute 'geo_to_h3'